# Notebook 1: sources, retrieval and profiling

This notebook walks through the first half of the project: which sources I used, how I checked the
download was complete, and what I found when I profiled the raw green taxi data.

It reads the files that `python run_all.py` puts in `data/raw/` (including the zone shapes file), so run
the pipeline first. It only reads raw files, it never writes or changes them.

## 1. Business questions and where the answers live

The client (hypothetical) is TLC's green taxi program team. Green taxis (Street Hail Liveries, also
called Boro Taxis) were created to serve street hails outside the core of Manhattan. TLC's green cab
page says they "may not operate in the Hail Exclusionary Zone, south of West 110th St and East 96th St",
and TLC's notice for the Boro Taxi decal rule adds the LaGuardia and JFK airports to that zone.

| # | Question from the program team | Information needed | Source |
|---|---|---|---|
| Q1 | How much green taxi service is there each month, and which way is it moving? | trips per month | TLC green trip files |
| Q2 | What share of street hails start in the area green taxis are allowed to serve? | pickup zone, hail or dispatch | trip files + taxi zone lookup + TLC rule |
| Q3 | Where and when do street hails get coded to the exclusionary zone or airports? | pickup zone, dropoff zone, pickup hour | trip files + zone lookup + zone shapes |
| Q4 | What does a Boro Zone street hail look like (how long, how fast)? | duration, distance | trip files |
| Q5 | Is the data complete and clean enough to act on? | an independent count of the same trips, rule checks | Open Data API `c5iv-bn4s` + validation |

The full source map (owner, grain, gaps) is in `docs/source_map.md`.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
import common as c

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
MONTHS = c.month_range(c.DEFAULT_START, c.DEFAULT_END)

## 2. Retrieval: did I get everything?

Two retrieval modes are used:

1. Files over HTTPS: 12 monthly parquet files and the zone lookup CSV from TLC's CloudFront bucket.
2. The NYC Open Data SODA API: TLC's own monthly pickup counts per taxi zone and industry (dataset
   `c5iv-bn4s`), and the taxi zone shapes (dataset `8meu-9t5y`).

SQL (SQLite) is used later for the model and the metrics, in notebook 2. It is not a retrieval mode here.

Before downloading anything, the pipeline checks that every requested month is published. For each file
I check that the bytes I saved equal the server's Content-Length, record a sha256, and compare the parquet
footer's row count with the rows I actually read. For the API I ask for `count(*)` first, then check the
rows I paged through add up to it. Each API page is saved exactly as it came back.

In [2]:
manifest = json.loads((c.RAW_DIR / "manifest.json").read_text())
rows = []
for m in MONTHS:
    e = manifest["trip_files"][m]
    df = pq.read_table(c.trip_file(m)).to_pandas()
    rows.append({
        "month": m,
        "bytes_saved": e["size_bytes"],
        "content_length": e["remote_content_length"],
        "sha256_now_matches": c.sha256_of(c.trip_file(m)) == e["sha256"],
        "parquet_footer_rows": e["parquet_rows"],
        "rows_read": len(df),
        "api_zone_rows": manifest["api"][m]["rows"],
        "api_count_star": manifest["api"][m]["expected_rows"],
    })
retrieval = pd.DataFrame(rows)
retrieval

,month,bytes_saved,content_length,sha256_now_matches,parquet_footer_rows,rows_read,api_zone_rows,api_count_star
0,2025-07,1188380,1188380,True,48205,48205,239,239
1,2025-08,1149707,1149707,True,46306,46306,228,228
2,2025-09,1209831,1209831,True,48893,48893,233,233
3,2025-10,1205685,1205685,True,49416,49416,230,230
4,2025-11,1164775,1164775,True,46912,46912,230,230
5,2025-12,1171145,1171145,True,48236,48236,228,228
6,2026-01,991656,991656,True,40272,40272,232,232
7,2026-02,920753,920753,True,37373,37373,229,229
8,2026-03,1082530,1082530,True,44208,44208,230,230
9,2026-04,1075896,1075896,True,44238,44238,234,234


In [3]:
checks = {
    "all 12 months present": len(retrieval) == 12,
    "bytes saved == Content-Length": (retrieval.bytes_saved == retrieval.content_length).all(),
    "checksums still match": retrieval.sha256_now_matches.all(),
    "rows read == parquet footer": (retrieval.rows_read == retrieval.parquet_footer_rows).all(),
    "API rows == API count(*)": (retrieval.api_zone_rows == retrieval.api_count_star).all(),
}
for name, ok in checks.items():
    print(f"{name:32s} {bool(ok)}")
print("total raw rows:", retrieval.rows_read.sum())

all 12 months present            True
bytes saved == Content-Length    True
checksums still match            True
rows read == parquet footer      True
API rows == API count(*)         True
total raw rows: 543143


Raw files are kept exactly as downloaded in `data/raw/` (gitignored because they are large and public).
The manifest with sizes and checksums is committed as `outputs/raw_manifest.json`, so anyone rerunning
can tell if TLC has republished a file since my run.

## 3. The zone lookup and the service area rule

In [4]:
zones = pd.read_csv(c.ZONE_CSV, keep_default_na=False)  # keep "N/A" as text
print(len(zones), "zones")
zones.service_zone.value_counts()

265 zones


service_zone
Boro Zone      205
Yellow Zone     55
Airports         2
N/A              2
EWR              1
Name: count, dtype: int64

In [5]:
zones[zones.service_zone.isin(["Airports", "EWR", "N/A"])]

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
131,132,Queens,JFK Airport,Airports
137,138,Queens,LaGuardia Airport,Airports
263,264,Unknown,N/A,N/A
264,265,N/A,Outside of NYC,N/A


The lookup already splits zones into "Boro Zone" (where green cabs can take hails), "Yellow Zone"
(Manhattan below 96th/110th), "Airports" and "EWR". So the permitted area is a zone-level mapping I can
join to.

The problem is that zones are big. A pickup coded to the Central Park zone could be on Central Park North
(110th St, which is the line itself) or at 72nd St. To see which Yellow Zone zones sit on the line,
the pipeline pulls the zone shapes from the NYC Open Data API (dataset `8meu-9t5y`), and here I check which
Yellow Zone / airport polygons touch a Boro Zone polygon.

In [6]:
from shapely.geometry import shape

# downloaded by stage1_ingest.py, checksum recorded in the manifest
print("shapes file matches manifest checksum:", c.sha256_of(c.ZONE_SHAPES) == manifest["zone_shapes"]["sha256"])
geo = json.loads(c.ZONE_SHAPES.read_text())
print(len(geo["features"]), "features in the shapes file")

shapes = {}
for f in geo["features"]:
    zid = int(float(f["properties"]["locationid"]))
    s = shape(f["geometry"]).buffer(0)
    shapes[zid] = shapes[zid].union(s) if zid in shapes else s

sz = dict(zip(zones.LocationID, zones.service_zone))
name = dict(zip(zones.LocationID, zones.Zone))
boro_ids = [z for z in shapes if sz.get(z) == "Boro Zone"]
touching = []
for z in sorted(shapes):
    if sz.get(z) == "Yellow Zone":
        nbrs = [b for b in boro_ids if shapes[z].distance(shapes[b]) < 0.0005]
        if nbrs:
            touching.append({"location_id": z, "zone": name[z], "touches": ", ".join(name[b] for b in nbrs)})
touching = pd.DataFrame(touching)
touching

shapes file matches manifest checksum: True
263 features in the shapes file


,location_id,zone,touches
0,24,Bloomingdale,"Central Harlem, Morningside Heights"
1,43,Central Park,"Central Harlem, East Harlem South"
2,194,Randalls Island,"East Harlem North, Mott Haven/Port Morris"
3,236,Upper East Side North,East Harlem South
4,262,Yorkville East,East Harlem South
5,263,Yorkville West,East Harlem South


In [7]:
print("boundary set in common.py:", sorted(c.BOUNDARY_ZONES))
print("matches the shapes:", set(touching.location_id) == c.BOUNDARY_ZONES)

boundary set in common.py: [24, 43, 194, 236, 262, 263]
matches the shapes: True


These six zones are what I call **boundary** zones. JFK and LaGuardia also touch Boro Zone polygons,
but the rule names the airports themselves, so I treat an airport pickup as clearly inside the
exclusionary zone. Newark is outside the city, so a street hail there is also clearly outside the
permitted area.

## 4. Profiling the raw trips

In [8]:
frames = []
for m in MONTHS:
    d = pq.read_table(c.trip_file(m)).to_pandas()
    d["file_month"] = m
    frames.append(d)
raw = pd.concat(frames, ignore_index=True)
print(raw.shape)
raw.dtypes

(543143, 23)


VendorID                          int32
lpep_pickup_datetime     datetime64[us]
lpep_dropoff_datetime    datetime64[us]
store_and_fwd_flag                  str
RatecodeID                      float64
PULocationID                      int32
DOLocationID                      int32
passenger_count                 float64
trip_distance                   float64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
ehail_fee                       float64
improvement_surcharge           float64
total_amount                    float64
payment_type                    float64
trip_type                       float64
congestion_surcharge            float64
cbd_congestion_fee              float64
file_month                          str
request_source                      str
dtype: object

In [9]:
# schema drift: which columns appear in which months
cols = {m: set(f.columns) for m, f in zip(MONTHS, frames)}
extra = {m: sorted(cols[m] - set(c.DOCUMENTED_COLUMNS) - {"file_month"}) for m in MONTHS}
{m: e for m, e in extra.items() if e}

{'2026-06': ['request_source']}

In [10]:
raw.isna().sum().sort_values(ascending=False).head(8)

ehail_fee               543143
request_source          536672
trip_type                67796
store_and_fwd_flag       67788
RatecodeID               67788
congestion_surcharge     67788
passenger_count          67788
payment_type             67788
dtype: int64

About one row in eight has no `trip_type`, `RatecodeID`, `passenger_count` or `payment_type`. The next
cell shows which vendor these rows come from.

In [11]:
raw["trip_type_label"] = raw.trip_type.map({1: "1 street-hail", 2: "2 dispatch"}).fillna("missing")
pd.crosstab(raw.VendorID, raw.trip_type_label, margins=True)

trip_type_label,1 street-hail,2 dispatch,missing,All
VendorID,,,,
1,47278,1735,831,49844
2,400565,25769,19014,445348
6,0,0,47951,47951
All,447843,27504,67796,543143


In [12]:
jun = frames[-1]
pd.crosstab(jun.request_source.fillna("null"), jun.trip_type.map({1: "1 street-hail", 2: "2 dispatch"}).fillna("missing"))

trip_type,1 street-hail,2 dispatch,missing
request_source,,,
A,0,0,6471
null,35504,2187,1


All of vendor 6's rows (Myle Technologies in the data dictionary) have no trip_type, plus some rows from
vendor 2. In June 2026 a new column `request_source` shows up with the value "A" on exactly these
rows. It's not in the data dictionary (dated March 18, 2025), so I don't know what "A" means. I did not
guess hail or dispatch for these trips. They are labelled "unclassified" and kept out of the KPI
denominator, and the share of them is reported as its own metric.

In [13]:
dur = (raw.lpep_dropoff_datetime - raw.lpep_pickup_datetime).dt.total_seconds() / 60
pickup_month = raw.lpep_pickup_datetime.dt.strftime("%Y-%m")
profile = pd.Series({
    "pickup outside file month": (pickup_month != raw.file_month).sum(),
    "  of which year 2008/2009": (raw.lpep_pickup_datetime.dt.year < 2010).sum(),
    "dropoff before pickup": (dur < 0).sum(),
    "zero duration": (dur == 0).sum(),
    "duration > 3 h": (dur > 180).sum(),
    "zero distance": (raw.trip_distance == 0).sum(),
    "distance > 100 mi": (raw.trip_distance > 100).sum(),
    "negative total_amount": (raw.total_amount < 0).sum(),
    "pickup zone 264/265": raw.PULocationID.isin([264, 265]).sum(),
})
profile

pickup outside file month      205
  of which year 2008/2009        4
dropoff before pickup            8
zero duration                  416
duration > 3 h                2279
zero distance                17866
distance > 100 mi              161
negative total_amount         1549
pickup zone 264/265           1773
dtype: int64

In [14]:
dur.describe(percentiles=[0.01, 0.5, 0.99, 0.999]).round(2)

count    543143.00
mean         21.29
std          79.11
min        -687.00
1%            0.07
50%          13.05
99%          82.88
99.9%      1410.00
max        2456.37
dtype: float64

In [15]:
long = raw[dur > 180]
print("median duration of >3h trips (min):", round(dur[dur > 180].median(), 1))
print("median distance of >3h trips vs all trips:", long.trip_distance.median(), raw.trip_distance.median())

median duration of >3h trips (min): 1392.8
median distance of >3h trips vs all trips: 2.9 2.08


The long trips are mostly about 23 hours but with a normal distance and fare, which looks like the meter
was not closed properly. The trip probably happened, but its duration is wrong, so these are WARN (kept
for counts, left out of duration and speed).

Next, the negative totals. My guess was that they are reversals of a real trip, so I checked if each
negative row has a positive twin with the same vendor, times and zones.

In [16]:
key = ["VendorID", "lpep_pickup_datetime", "lpep_dropoff_datetime", "PULocationID", "DOLocationID"]
neg = raw[raw.total_amount < 0]
pos_keys = raw[raw.total_amount > 0][key].drop_duplicates()
twins = neg.merge(pos_keys, on=key, how="inner")
print("negative rows:", len(neg), "| with a positive twin:", len(twins))
print("exact duplicate rows:", raw.duplicated().sum())
print("payment_type of negative rows:")
print(neg.payment_type.value_counts(dropna=False))
no_twin = neg.merge(pos_keys, on=key, how="left", indicator=True).query("_merge == 'left_only'")
same_start = no_twin.merge(raw[raw.total_amount > 0][["VendorID", "lpep_pickup_datetime"]].drop_duplicates(),
                           on=["VendorID", "lpep_pickup_datetime"])
print("no exact twin:", len(no_twin), "| of those, same vendor and pickup time as a positive row:", len(same_start))

negative rows: 1549 | with a positive twin: 1526


exact duplicate rows: 0
payment_type of negative rows:
payment_type
3.0    1053
4.0     400
1.0      53
2.0      42
NaN       1
Name: count, dtype: int64


no exact twin: 23 | of those, same vendor and pickup time as a positive row: 22


Almost every negative row has a positive twin, and there are no exact duplicates. So the "duplicates" in
this data are reversal pairs. If I counted both rows, the trip would be counted twice. The negative row is a
FAIL (excluded from metrics) and the original stays. The 23 rows without an exact twin are also excluded,
since a negative total can't be a normal trip, and 22 of them share vendor and pickup time with a
positive row anyway.

## 5. Cross-check with TLC's own published counts (API)

TLC publishes monthly Green Cab pickups per zone on Open Data. That dataset leaves out zones 264/265, so I
compare it with my file rows that have a known pickup zone.

In [17]:
api = []
for m in MONTHS:
    for page in range(1, len(manifest["api"][m]["page_sha256"]) + 1):
        for r in json.loads(c.api_page_file(m, page).read_text()):
            api.append({"file_month": m, "location_id": int(r["locationid"]), "trip_count": int(r["trip_count"])})
api = pd.DataFrame(api)
ours = raw[~raw.PULocationID.isin([264, 265])].groupby("file_month").size().rename("file_rows_known_zone")
recon = api.groupby("file_month").trip_count.sum().rename("api_pickups").to_frame().join(ours)
recon["gap_pct"] = (100 * (recon.file_rows_known_zone / recon.api_pickups - 1)).round(2)
recon

,api_pickups,file_rows_known_zone,gap_pct
file_month,,,
2025-07,48118,48044,-0.15
2025-08,46124,46111,-0.03
2025-09,48743,48747,0.01
2025-10,49244,49241,-0.01
2025-11,46803,46796,-0.01
2025-12,48094,48105,0.02
2026-01,40141,40142,0.00
2026-02,37263,37249,-0.04
2026-03,44064,44068,0.01


Eleven months agree with TLC's number to within 0.15%. May 2026 is short by about 2%. The next cell
looks at which vendor is missing.

In [18]:
may = raw[(raw.file_month == "2026-05") & (raw.lpep_pickup_datetime.dt.strftime("%Y-%m") == "2026-05")]
by_day = pd.crosstab(may.lpep_pickup_datetime.dt.day, may.VendorID)
by_day.tail(9)

VendorID,1,2,6
lpep_pickup_datetime,,,
23,79,1066,85
24,54,976,70
25,102,1073,26
26,109,1120,1
27,119,1041,0
28,135,1224,0
29,130,1209,0
30,63,1045,0
31,65,1008,0


In [19]:
last_day = (raw[raw.lpep_pickup_datetime.dt.strftime("%Y-%m") == raw.file_month]
            .groupby(["file_month", "VendorID"]).lpep_pickup_datetime.max().dt.day.unstack())
last_day

VendorID,1,2,6
file_month,,,
2025-07,31,31,31
2025-08,31,31,31
2025-09,30,30,30
2025-10,31,31,31
2025-11,30,30,30
2025-12,31,31,31
2026-01,31,31,31
2026-02,28,28,28
2026-03,31,31,31


Vendor 6 stops on May 26 in the May file. Every other month it reports to the last day. The missing
days (May 27 to 31) are roughly the size of the gap against TLC's aggregate, so the May file looks
incomplete for that vendor. The pipeline picks this up on its own with two checks
(`vendor_6_reports_to_month_end` and the API gap) and publishes May with a WARN status.
It does not hold May back, because vendor 6 trips have no trip_type and so never enter the KPI anyway.
Only the volume numbers for May are affected. When I checked on 2026-09-25, the May file on the TLC
site still had the same size and a last-modified date of 2026-06-26, so it has not been re-posted.

## 6. What the data says about the problem

Before building metrics, I wanted to see the problem in the raw data: how volume moves and where pickups
happen by request type.

In [20]:
merged = raw.merge(zones, left_on="PULocationID", right_on="LocationID", how="left")
merged["request"] = merged.trip_type.map({1: "street_hail", 2: "dispatch"}).fillna("unclassified")
pd.crosstab(merged.service_zone, merged.request, margins=True)

request,dispatch,street_hail,unclassified,All
service_zone,,,,
Airports,258,207,22,487
Boro Zone,26229,417584,59869,503682
EWR,9,1,0,10
N/A,946,748,79,1773
Yellow Zone,62,29303,7826,37191
All,27504,447843,67796,543143


In [21]:
(pd.crosstab(merged.service_zone, merged.request, normalize="columns") * 100).round(2)

request,dispatch,street_hail,unclassified
service_zone,,,
Airports,0.94,0.05,0.03
Boro Zone,95.36,93.24,88.31
EWR,0.03,0.00,0.00
N/A,3.44,0.17,0.12
Yellow Zone,0.23,6.54,11.54


In [22]:
hails_yellow = merged[(merged.request == "street_hail") & (merged.service_zone == "Yellow Zone")]
hails_yellow.Zone.value_counts().head(8)

Zone
Central Park             22300
Upper East Side North     3423
Bloomingdale              3088
Yorkville West             395
Yorkville East              42
Randalls Island             21
Manhattan Valley            11
Upper West Side North        9
Name: count, dtype: int64

What I take from this before any cleaning:

- Most street hails are in the Boro Zone, but a few percent are coded to the Yellow Zone.
  That is the number the program team would care about.
- Almost all of those Yellow Zone hails are in a handful of zones on the 96th/110th line, led by Central Park.
  Midtown and downtown barely show up.
- Dispatch trips (which are allowed anywhere) are almost never in the Yellow Zone, while the unclassified
  rows are in the Yellow Zone much more often (11.54% of them). So the unclassified rows look different
  from the rest and I can't just fold them into hail or dispatch.

Exact values after validation are in notebook 2 and `outputs/evidence_table.md`.